# Imports

In [14]:
#install Vader through "pip install vaderSentiment" in terminal
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
df = pd.read_csv('../../Data/2. IntermediateData/df_binarized_hard.csv')

In [15]:
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,movie_id.1,title,...,credit_pos,year,id,imdbid,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,3,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,3,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137057,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0
137058,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0
137059,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0
137060,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,m615,young frankenstein,...,?,1974,1077,72431,0,8.0,176404,106,Comedy,0


In [16]:
df=df.drop(['movie_id.1'], axis=1)

# Calculating sentiment score with VADER

In [17]:
# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get polarity scores
def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

# Apply sentiment analysis to each line of dialogue
df['sentiment'] = df['text'].apply(get_sentiment)

# 1. Mean sentiment polarity by gender overall
mean_sentiment_by_gender = df.groupby('gender')['sentiment'].mean()

# 2. Mean sentiment polarity by gender split by Bechdel score
mean_sentiment_by_gender_bechdel = df.groupby(['bechdel_score', 'gender'])['sentiment'].mean()

# To present it more clearly, you can unstack the results
mean_sentiment_split = mean_sentiment_by_gender_bechdel.unstack()

print("Overall mean sentiment by gender:")
print(mean_sentiment_by_gender)

print("\nMean sentiment by gender and Bechdel score:")
print(mean_sentiment_split)

Overall mean sentiment by gender:
gender
f    0.058220
m    0.048871
Name: sentiment, dtype: float64

Mean sentiment by gender and Bechdel score:
gender                f         m
bechdel_score                    
0              0.058090  0.044079
1              0.058289  0.055099


In [18]:
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,title,gender,...,year,id,imdbid,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar,sentiment
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,10 things i hate about you,f,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,10 things i hate about you,m,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,10 things i hate about you,f,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.4404
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,10 things i hate about you,m,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.2263
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,10 things i hate about you,f,...,1999,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137057,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',young frankenstein,f,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,-0.0772
137058,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,young frankenstein,m,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,0.0000
137059,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',young frankenstein,f,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,0.0000
137060,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,young frankenstein,m,...,1974,1077,72431,0,8.0,176404,106,Comedy,0,0.0000


In [24]:
# Categorize sentiment first
df['sentiment_category'] = df['sentiment'].apply(
    lambda x: 'Positive' if x >= 0.05 
    else 'Negative' if x <= -0.05 
    else 'Neutral'
)

# Print text for each category
print("="*50 + "\nFIRST 40 POSITIVE TEXTS\n" + "="*50)
for text in df[df['sentiment_category'] == 'Positive']['text'].head(100):
    print(text)
    print("-"*50)  # Separator between entries

print("\n" + "="*50 + "\nFIRST 40 NEUTRAL TEXTS\n" + "="*50) 
for text in df[df['sentiment_category'] == 'Neutral']['text'].head(40):
    print(text)
    print("-"*50)

print("\n" + "="*50 + "\nFIRST 40 NEGATIVE TEXTS\n" + "="*50)
for text in df[df['sentiment_category'] == 'Negative']['text'].head(40):
    print(text)
    print("-"*50)

FIRST 40 POSITIVE TEXTS
I hope so.
--------------------------------------------------
She okay?
--------------------------------------------------
Wow
--------------------------------------------------
Okay -- you're gonna need to learn how to lie.
--------------------------------------------------
I'm kidding.  You know how sometimes you just become this "persona"?  And you don't know how to quit?
--------------------------------------------------
What good stuff?
--------------------------------------------------
I figured you'd get to the good stuff eventually.
--------------------------------------------------
Thank God!  If I had to hear one more story about your coiffure...
--------------------------------------------------
Me.  This endless ...blonde babble. I'm like, boring myself.
--------------------------------------------------
Then Guillermo says, "If you go any lighter, you're gonna look like an extra on 90210."
--------------------------------------------------
Well, no.